# 05 — Row Mutations: upsert, delete, overwrite

PyIceberg 0.9+ supports three row-level mutation patterns:
- `upsert(df, join_cols=...)` — merge by key (insert new, update existing).
- `delete(delete_filter=...)` — drop rows matching a predicate.
- `overwrite(df, overwrite_filter=...)` — atomic delete-and-insert under filter.

Each call produces a new snapshot.

**Prerequisites:** Run `01_write_iceberg_polars.ipynb` first.
This notebook does NOT depend on 03/04 — it ignores any schema/partition changes.

In [1]:
import datetime

import polars as pl
from pyiceberg.expressions import EqualTo, GreaterThan
from pyiceberg.io.pyarrow import schema_to_pyarrow

import os

from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

table = catalog.load_table(("demo", "events"))
table.refresh()
print("Start snapshot count:", len(table.history()))
print("Start row count:    ", len(pl.from_arrow(table.scan().to_arrow())))

Connected to catalog: nessie
Start snapshot count: 1
Start row count:     10


## 1. `upsert()` — merge by `event_id`

Three rows: one update (`e001` → new amount), two inserts (`e200`, `e201`).

In [2]:
UTC = datetime.timezone.utc

upsert_df = pl.DataFrame(
    {
        "event_id":   ["e001", "e200", "e201"],
        "user_id":    [101, 301, 302],
        "event_kind": ["click", "purchase", "view"],
        "amount":     [1.50, 19.99, 0.0],
        "ts": [
            datetime.datetime(2024, 3, 1,  9, 0, 0, tzinfo=UTC),
            datetime.datetime(2024, 3, 1, 10, 0, 0, tzinfo=UTC),
            datetime.datetime(2024, 3, 1, 11, 0, 0, tzinfo=UTC),
        ],
    }
)

# 03 may have renamed event_type → event_kind. Align the column name first,
# then reorder + cast to the live table schema.
schema_names = {f.name for f in table.schema().fields}
if "event_type" in schema_names and "event_kind" not in schema_names:
    upsert_df = upsert_df.rename({"event_kind": "event_type"})

field_names = [f.name for f in table.schema().fields]
result = table.upsert(
    upsert_df.select(field_names).to_arrow().cast(schema_to_pyarrow(table.schema())),
    join_cols=["event_id"],
)
table.refresh()

print("Rows updated:", result.rows_updated)
print("Rows inserted:", result.rows_inserted)
print("Snapshots:", len(table.history()))

Rows updated: 1
Rows inserted: 2
Snapshots: 1


## 2. `delete()` with a row filter

In [3]:
table.delete(delete_filter=GreaterThan("amount", 50.0))
table.refresh()

print("Snapshots:", len(table.history()))
print("Rows after delete:", len(pl.from_arrow(table.scan().to_arrow())))

Snapshots: 1
Rows after delete: 11


## 3. `overwrite()` with a filter — atomic replace-by-key

Useful when you want delete-and-insert semantics in one commit.

In [4]:
replace_kind = "event_kind" if "event_kind" in {f.name for f in table.schema().fields} else "event_type"

replacement = pl.DataFrame(
    {
        "event_id":   ["e001"],
        "user_id":    [101],
        replace_kind:  ["click"],
        "amount":     [2.50],
        "ts":         [datetime.datetime(2024, 3, 2, 12, 0, 0, tzinfo=UTC)],
    }
)

field_names = [f.name for f in table.schema().fields]
table.overwrite(
    replacement.select(field_names).to_arrow().cast(schema_to_pyarrow(table.schema())),
    overwrite_filter=EqualTo("event_id", "e001"),
)
table.refresh()
print("Snapshots:", len(table.history()))

Snapshots: 1


## 4. Final state

In [5]:
df = pl.from_arrow(table.scan().to_arrow()).sort("ts")
print(f"Final row count: {len(df)}")
df

Final row count: 11


ts,event_id,user_id,event_kind,amount
"datetime[μs, UTC]",str,i64,str,f64
2024-01-15 11:00:00 UTC,"""e002""",102,"""view""",5.99
2024-01-16 09:00:00 UTC,"""e003""",103,"""click""",0.0
2024-01-16 14:00:00 UTC,"""e004""",101,"""purchase""",49.99
2024-01-17 08:00:00 UTC,"""e005""",104,"""view""",0.0
2024-01-18 09:00:00 UTC,"""e006""",105,"""click""",0.0
…,…,…,…,…
2024-02-01 09:30:00 UTC,"""e101""",201,"""view""",0.0
2024-02-01 10:00:00 UTC,"""e102""",202,"""click""",0.0
2024-03-01 10:00:00 UTC,"""e200""",301,"""purchase""",19.99


---
**Next:** [06_metadata_inspection.ipynb](06_metadata_inspection.ipynb) — query the snapshots, files, manifests this notebook created.
Also: [09_snapshot_management.ipynb](09_snapshot_management.ipynb) for rollback to any of these snapshots.